# Private AI Agent - Production Colab Runtime (Ollama + Mistral)

This notebook is a clean, secure runbook to run `private-ai-agent` in Google Colab (including VS Code attached to Colab kernel).

What this notebook does:
1. Validates runtime and repository context.
2. Clones the repository securely (no hardcoded personal tokens).
3. Installs the package into the current notebook kernel.
4. Starts Ollama and pulls `mistral-small3.2` + `nomic-embed-text`.
5. Writes `.env`, ingests documents, runs a QA smoke test.
6. Proves Chroma retrieval is used at inference time.
7. Renders the LangGraph graph as PNG.
8. Runs a 10-question latency + quality benchmark (p50/p95).


In [ ]:
# Confirm runtime and baseline details.
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Version:", sys.version)
print("CWD:", os.getcwd())
print("In Colab:", "google.colab" in sys.modules or Path("/content").exists())


In [ ]:
# Configure repository target. Update REPO_URL before first run.
from pathlib import Path

REPO_URL = "https://github.com/<ORG_OR_USER>/private-ai-agent.git"
REPO_DIR = Path("/content/private-ai-agent")
GIT_BRANCH = "main"

print("REPO_URL:", REPO_URL)
print("REPO_DIR:", REPO_DIR)
print("GIT_BRANCH:", GIT_BRANCH)


In [ ]:
# Clone repository securely.
# - Uses Colab secret GITHUB_TOKEN when available.
# - Falls back to normal clone for public repositories.
import base64
import subprocess

if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}. Skipping clone.")
else:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None

    if token:
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        subprocess.run(
            [
                "git",
                "-c",
                f"http.extraheader=AUTHORIZATION: basic {auth}",
                "clone",
                "--branch",
                GIT_BRANCH,
                REPO_URL,
                str(REPO_DIR),
            ],
            check=True,
        )
        print("Private clone completed with token from Colab Secrets.")
    else:
        subprocess.run(
            ["git", "clone", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
        print("Clone completed without token (public repo mode).")


In [ ]:
# Enter repository and install package into THIS notebook kernel.
%cd /content/private-ai-agent
!git status --short
%pip install -qU pip setuptools wheel
%pip install -q -e .


In [ ]:
# Verify importability in the current kernel.
# If editable install path is not yet active, insert src as a runtime fallback.
import importlib.util
import os
import sys

spec = importlib.util.find_spec("private_agent")
if spec is None:
    sys.path.insert(0, "/content/private-ai-agent/src")
    spec = importlib.util.find_spec("private_agent")

assert spec is not None, "private_agent not importable. Restart kernel and rerun install cell."

import private_agent
print("private_agent module:", private_agent.__file__)
print("python executable:", sys.executable)
print("cwd:", os.getcwd())


In [ ]:
# Install Ollama if missing, then start the server in background.
!which ollama || (curl -fsSL https://ollama.com/install.sh | sh)
!pkill -f "ollama serve" || true
!nohup ollama serve > /tmp/ollama.log 2>&1 &

import time
import subprocess

time.sleep(5)
subprocess.run(
    ["bash", "-lc", "curl -sSf http://127.0.0.1:11434/api/tags > /dev/null"],
    check=True,
)
print("Ollama API is healthy at http://127.0.0.1:11434")


In [ ]:
# Pull required models for this project.
!ollama pull mistral-small3.2
!ollama pull nomic-embed-text
!ollama list


In [ ]:
# Write .env for Ollama runtime profile.
from pathlib import Path

env_lines = [
    "PRIVATE_AGENT_LLM_PROVIDER=ollama",
    "PRIVATE_AGENT_LLM_MODEL=mistral-small3.2",
    "PRIVATE_AGENT_LLM_TEMPERATURE=0",
    "PRIVATE_AGENT_LLM_TIMEOUT_SECONDS=180",
    "",
    "PRIVATE_AGENT_EMBEDDING_PROVIDER=ollama",
    "PRIVATE_AGENT_EMBEDDING_MODEL=nomic-embed-text",
    "",
    "PRIVATE_AGENT_DATA_DIR=./data",
    "PRIVATE_AGENT_VECTOR_DB_DIR=./chroma_db",
    "PRIVATE_AGENT_COLLECTION_NAME=enterprise_knowledge",
    "",
    "PRIVATE_AGENT_CHUNK_SIZE=700",
    "PRIVATE_AGENT_CHUNK_OVERLAP=120",
    "PRIVATE_AGENT_RETRIEVER_K=4",
    "",
    "PRIVATE_AGENT_LOG_LEVEL=INFO",
]

env_path = Path(".env")
env_path.write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print(f"Wrote {env_path.resolve()}")
print(env_path.read_text(encoding="utf-8"))


In [ ]:
# Validate loaded settings from .env.
!python -c "from private_agent.config import load_settings; s=load_settings(); print(s)"


In [ ]:
# Rebuild Chroma index and run a smoke test question.
!python -m private_agent.ingest --rebuild
!python -m private_agent.main --question "Who approves a £12,000 invoice?"


In [ ]:
# Prove Chroma retrieval is used for answer generation.
from private_agent.config import load_settings
from private_agent.graph import build_graph
import chromadb

settings = load_settings()
client = chromadb.PersistentClient(path=str(settings.vector_db_dir))
collection = client.get_collection(settings.collection_name)
print("vector_db_dir:", settings.vector_db_dir)
print("collection:", settings.collection_name)
print("vector_count:", collection.count())

app = build_graph(settings)
result = app.invoke(
    {
        "question": "Who approves a £12,000 invoice?",
        "context": [],
        "answer": "",
    }
)

print("retrieved_docs:", len(result["context"]))
for i, doc in enumerate(result["context"], 1):
    preview = doc.page_content[:140].replace("\n", " ")
    print(f"[{i}] source={doc.metadata.get('source')} | {preview}")

print("\nfinal_answer:", result["answer"])


In [ ]:
# Render the LangGraph workflow as PNG.
from IPython.display import Image, display
from private_agent.config import load_settings
from private_agent.graph import build_graph

app = build_graph(load_settings())
png_bytes = app.get_graph().draw_mermaid_png()
display(Image(png_bytes))

with open("private_agent_graph.png", "wb") as f:
    f.write(png_bytes)

print("Saved: /content/private-ai-agent/private_agent_graph.png")


In [ ]:
# Benchmark latency and simple quality checks across 10 questions.
import csv
import json
import math
import statistics
import time
from pathlib import Path
from urllib.request import urlopen

from private_agent.config import load_settings
from private_agent.graph import build_graph

BENCHMARK_SET = [
    ("Who approves a £12,000 invoice?", ["cfo"]),
    ("Who can approve invoices below £1,000?", ["finance associate"]),
    ("What happens to duplicate invoices?", ["manual review"]),
    ("Can we auto-pay invoices without a PO number?", ["must not", "paid"]),
    ("How many annual leave days are employees entitled to?", ["25", "annual leave"]),
    ("When is medical evidence required for sick leave?", ["7", "medical evidence"]),
    ("Should the AI make final HR decisions?", ["must not", "final"]),
    ("What priority is payment failure affecting active customers?", ["high priority"]),
    ("How should security incidents be handled?", ["urgent", "escalated"]),
    ("Do refund requests above £500 need human approval?", ["human approval"]),
]


def percentile(values, p):
    values = sorted(values)
    if not values:
        return float("nan")
    if len(values) == 1:
        return values[0]
    k = (len(values) - 1) * (p / 100.0)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return values[f]
    return values[f] * (c - k) + values[c] * (k - f)


def check_ollama_health():
    with urlopen("http://127.0.0.1:11434/api/tags", timeout=5) as r:
        if r.status != 200:
            raise RuntimeError(f"Ollama health check failed: HTTP {r.status}")


def run_benchmark(max_retries=1, warmup_runs=2):
    settings = load_settings()
    if not settings.vector_db_dir.exists():
        raise RuntimeError(f"Vector DB missing at {settings.vector_db_dir}. Run ingest first.")

    check_ollama_health()
    app = build_graph(settings)

    for i in range(min(warmup_runs, len(BENCHMARK_SET))):
        q, _ = BENCHMARK_SET[i]
        _ = app.invoke({"question": q, "context": [], "answer": ""})

    rows = []
    start_all = time.perf_counter()

    for idx, (question, must_contain) in enumerate(BENCHMARK_SET, start=1):
        for attempt in range(max_retries + 1):
            t0 = time.perf_counter()
            try:
                result = app.invoke({"question": question, "context": [], "answer": ""})
                latency_s = time.perf_counter() - t0
                answer = str(result.get("answer", "")).strip()
                answer_lower = answer.lower()
                quality_pass = all(token in answer_lower for token in must_contain)

                rows.append(
                    {
                        "id": idx,
                        "question": question,
                        "latency_s": round(latency_s, 3),
                        "quality_pass": quality_pass,
                        "status": "ok",
                        "error": "",
                        "answer_preview": answer[:180].replace("\n", " "),
                    }
                )
                break
            except Exception as exc:
                if attempt < max_retries:
                    time.sleep(1.5)
                    continue
                rows.append(
                    {
                        "id": idx,
                        "question": question,
                        "latency_s": None,
                        "quality_pass": False,
                        "status": "failed",
                        "error": str(exc)[:240],
                        "answer_preview": "",
                    }
                )

    total_s = time.perf_counter() - start_all
    ok_latencies = [r["latency_s"] for r in rows if r["status"] == "ok" and r["latency_s"] is not None]

    ok_count = sum(1 for r in rows if r["status"] == "ok")
    fail_count = len(rows) - ok_count
    quality_pass_count = sum(1 for r in rows if r["quality_pass"])

    summary = {
        "samples": len(rows),
        "success_count": ok_count,
        "failure_count": fail_count,
        "error_rate": round(fail_count / len(rows), 3),
        "quality_pass_rate": round(quality_pass_count / len(rows), 3),
        "p50_latency_s": round(percentile(ok_latencies, 50), 3) if ok_latencies else None,
        "p95_latency_s": round(percentile(ok_latencies, 95), 3) if ok_latencies else None,
        "mean_latency_s": round(statistics.mean(ok_latencies), 3) if ok_latencies else None,
        "min_latency_s": round(min(ok_latencies), 3) if ok_latencies else None,
        "max_latency_s": round(max(ok_latencies), 3) if ok_latencies else None,
        "throughput_qps": round(ok_count / total_s, 3) if total_s > 0 else None,
        "total_wall_time_s": round(total_s, 3),
        "llm_provider": settings.llm_provider,
        "llm_model": settings.llm_model,
        "embedding_provider": settings.embedding_provider,
        "embedding_model": settings.embedding_model,
    }

    for row in rows:
        print(f"[{row['id']:02d}] {row['status']:6} | {str(row['latency_s']).rjust(6)}s | quality={row['quality_pass']} | {row['question']}")

    print("\nSummary:")
    print(json.dumps(summary, indent=2))

    out_path = Path("benchmark_results.csv")
    with out_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSaved benchmark rows to: {out_path.resolve()}")

run_benchmark()


## Production Notes

- Keep `.env` values environment-specific (dev/stage/prod).
- Never commit secrets (`.env`, PATs, API keys) to git.
- Re-run ingestion whenever source policy documents change.
- For stable benchmarking, use the same Colab runtime type and run warmup before measurement.
